In [1]:
#cell 1

import json
import pandas as pd
from pathlib import Path

# --- 1. 경로 설정 ---
try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path.cwd()
    print(f"PROJECT_ROOT not set, defaulting to CWD: {PROJECT_ROOT}")

# Fashionpedia 원본 데이터 경로 (환경에 맞게 수정)
FP_DATA_DIR = PROJECT_ROOT / "data" / "Fashionpedia"
# [!!] 이 파일이 훈련용 어노테이션 파일이 맞는지 확인하세요.
FP_TRAIN_JSON_PATH = FP_DATA_DIR / "annotations" / "instances_attributes_train2020.json"

print(f"Loading Fashionpedia (Shop) annotations from: {FP_TRAIN_JSON_PATH}")

# --- 2. JSON 파일 로드 ---
try:
    with open(FP_TRAIN_JSON_PATH, 'r') as f:
        data_fp_train = json.load(f)
    
    print("\nFashionpedia (Shop) 훈련 데이터 로드 성공!")

    # --- 3. 카테고리 목록(46종) 추출 및 DataFrame으로 변환 ---
    # 다음 작업(레이블 매핑)을 위해 원본 카테고리 목록을 확인합니다.
    categories_fp = data_fp_train.get('categories', [])
    
    if categories_fp:
        df_fp_categories = pd.DataFrame(categories_fp)
        
        print(f"\n--- Fashionpedia 원본 카테고리 목록 ({len(df_fp_categories)}종) ---")
        # 'name'과 'supercategory'를 함께 출력하여 매핑에 참고
        print(df_fp_categories[['id', 'name', 'supercategory']].to_string())
        
        # 다음 단계에서 사용할 변수 저장
        %store data_fp_train
        %store df_fp_categories
    else:
        print("\n[Error] 'categories' 키를 JSON 파일에서 찾을 수 없습니다.")

except FileNotFoundError:
    print(f"\n[Fatal Error] 파일을 찾을 수 없습니다: {FP_TRAIN_JSON_PATH}")
    print("Fashionpedia 데이터셋 경로를 확인하거나, 파일 이름을 (예: 'attributes_train2020.json')으로 수정해 주세요.")
except Exception as e:
    print(f"\n[Error] JSON 파일 로드 중 오류 발생: {e}")

PROJECT_ROOT not set, defaulting to CWD: /home/epistachio/projects/fashion_ai
Loading Fashionpedia (Shop) annotations from: /home/epistachio/projects/fashion_ai/data/Fashionpedia/annotations/instances_attributes_train2020.json

✅ Fashionpedia (Shop) 훈련 데이터 로드 성공!

--- Fashionpedia 원본 카테고리 목록 (46종) ---
    id                                     name   supercategory
0    0                            shirt, blouse       upperbody
1    1                 top, t-shirt, sweatshirt       upperbody
2    2                                  sweater       upperbody
3    3                                 cardigan       upperbody
4    4                                   jacket       upperbody
5    5                                     vest       upperbody
6    6                                    pants       lowerbody
7    7                                   shorts       lowerbody
8    8                                    skirt       lowerbody
9    9                                     coat       who

In [2]:
#cell 2

import pandas as pd

# --- 1. [Task 1]에서 저장한 변수 로드 ---
try:
    %store -r data_fp_train
    %store -r df_fp_categories
except Exception as e:
    print(f"[Fatal Error] '%store -r' failed. Task 1을 다시 실행해야 할 수도 있습니다: {e}")
    # Task 1에서 로드를 못했을 경우를 대비한 최소한의 복구 코드
    if 'df_fp_categories' not in locals():
         categories_fp = data_fp_train.get('categories', [])
         df_fp_categories = pd.DataFrame(categories_fp)

# --- 2. 우리가 사용할 최종 13개 카테고리 (Target) ---
TARGET_LABELS = [
    'short sleeve top',    # 0
    'long sleeve top',     # 1
    'short sleeve outwear',# 2
    'long sleeve outwear', # 3
    'vest',                # 4
    'sling',               # 5
    'shorts',              # 6
    'trousers',            # 7
    'skirt',               # 8
    'short sleeve dress',  # 9
    'long sleeve dress',   # 10
    'vest dress',          # 11
    'sling dress'          # 12
]
print(f"매핑 대상(Target) 13개 레이블이 정의되었습니다. (예: 0 = {TARGET_LABELS[0]})")

# --- 3. [핵심] 46종(FP) -> 13종(DF2) 매핑 딕셔너리 ---
# Fashionpedia의 'id'를 13종의 '인덱스(0~12)'로 변환합니다.
# `None`은 13종에 해당하지 않아 "무시"할 아이템입니다 (예: 'glasses', 'shoe', 'zipper').

fp_id_to_target_idx = {
    #--- Upperbody (상체) ---
    0: 1,  # 'shirt, blouse' -> 'long sleeve top' (1)
    1: 0,  # 'top, t-shirt, sweatshirt' -> 'short sleeve top' (0)
    2: 1,  # 'sweater' -> 'long sleeve top' (1)
    3: 3,  # 'cardigan' -> 'long sleeve outwear' (3)
    4: 3,  # 'jacket' -> 'long sleeve outwear' (3)
    5: 4,  # 'vest' -> 'vest' (4)
    #--- Lowerbody (하체) ---
    6: 7,  # 'pants' -> 'trousers' (7)
    7: 6,  # 'shorts' -> 'shorts' (6)
    8: 8,  # 'skirt' -> 'skirt' (8)
    #--- Wholebody (전신) ---
    9: 3,  # 'coat' -> 'long sleeve outwear' (3) (가장 유사)
    10: 9, # 'dress' -> 'short sleeve dress' (9) (기본값)
    11: None, # 'jumpsuit' -> 무시
    12: None, # 'cape' -> 무시
    #--- Head (머리) ---
    13: None, # 'glasses' -> 무시
    14: None, # 'hat' -> 무시
    15: None, # 'headband...' -> 무시
    #--- Neck (목) ---
    16: None, # 'tie' -> 무시
    #--- Arms/Hands (팔/손) ---
    17: None, # 'glove' -> 무시
    18: None, # 'watch' -> 무시
    #--- Waist (허리) ---
    19: None, # 'belt' -> 무시
    #--- Legs/Feet (다리/발) ---
    20: None, # 'leg warmer' -> 무시
    21: None, # 'tights, stockings' -> 무시
    22: None, # 'sock' -> 무시
    23: None, # 'shoe' -> 무시
    #--- Others (기타) ---
    24: None, # 'bag, wallet' -> 무시
    25: None, # 'scarf' -> 무시
    26: None, # 'umbrella' -> 무시
    #--- Garment Parts (부분) ---
    27: None, # 'hood' -> 무시
    28: None, # 'collar' -> 무시
    29: None, # 'lapel' -> 무시
    30: None, # 'epaulette' -> 무시
    31: None, # 'sleeve' -> 무시
    32: None, # 'pocket' -> 무시
    33: None, # 'neckline' -> 무시
    #--- Closures (여밈) ---
    34: None, # 'buckle' -> 무시
    35: None, # 'zipper' -> 무시
    #--- Decorations (장식) ---
    36: None, # 'applique' -> 무시
    37: None, # 'bead' -> 무시
    38: None, # 'bow' -> 무시
    39: None, # 'flower' -> 무시
    40: None, # 'fringe' -> 무시
    41: None, # 'ribbon' -> 무시
    42: None, # 'rivet' -> 무시
    43: None, # 'ruffle' -> 무시
    44: None, # 'sequin' -> 무시
    45: None, # 'tassel' -> 무시
}
print(f"Fashionpedia 46종 -> Target 13종 매핑 딕셔너리 생성 완료.")

# --- 4. 매핑 결과 검증 ---
# 이 매핑을 원본 카테고리 DataFrame에 적용
df_fp_categories['target_idx'] = df_fp_categories['id'].map(fp_id_to_target_idx)
# 'None'이 아닌 (즉, 13종에 매핑된) 카테고리만 필터링
mapped_categories = df_fp_categories.dropna(subset=['target_idx'])
mapped_categories['target_name'] = mapped_categories['target_idx'].apply(lambda x: TARGET_LABELS[int(x)])

print("\n--- [검증] 매핑 결과 (무시되지 않는 카테고리) ---")
print(mapped_categories[['id', 'name', 'target_name']].to_string())

# --- 5. 다음 단계를 위해 변수 저장 ---
%store fp_id_to_target_idx
%store TARGET_LABELS

print("\n매핑 규칙이 성공적으로 정의 및 저장")

매핑 대상(Target) 13개 레이블이 정의되었습니다. (예: 0 = short sleeve top)
Fashionpedia 46종 -> Target 13종 매핑 딕셔너리 생성 완료.

--- [검증] 매핑 결과 (무시되지 않는 카테고리) ---
    id                      name          target_name
0    0             shirt, blouse      long sleeve top
1    1  top, t-shirt, sweatshirt     short sleeve top
2    2                   sweater      long sleeve top
3    3                  cardigan  long sleeve outwear
4    4                    jacket  long sleeve outwear
5    5                      vest                 vest
6    6                     pants             trousers
7    7                    shorts               shorts
8    8                     skirt                skirt
9    9                      coat  long sleeve outwear
10  10                     dress   short sleeve dress
Stored 'fp_id_to_target_idx' (dict)
Stored 'TARGET_LABELS' (list)

✅ 매핑 규칙이 성공적으로 정의 및 저장되었습니다.


/tmp/ipykernel_443/1769427671.py:105: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mapped_categories['target_name'] = mapped_categories['target_idx'].apply(lambda x: TARGET_LABELS[int(x)])


In [3]:
#cell 3

import json
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm # 진행률 표시

# --- 1. [Task 1, 2]에서 저장한 변수 로드 ---
try:
    %store -r data_fp_train
    %store -r fp_id_to_target_idx
except Exception as e:
    print(f"[Fatal Error] '%store -r' failed. Task 1과 2를 다시 실행해야 합니다: {e}")
    raise e

# --- 2. 경로 설정 및 이미지 ID-경로 맵 생성 ---
# (Task 1 코드와 동일한 경로)
PROJECT_ROOT = Path.cwd()
FP_DATA_DIR = PROJECT_ROOT / "data" / "Fashionpedia"
FP_TRAIN_IMAGE_DIR = FP_DATA_DIR / "train" # 훈련 이미지 폴더

# 빠른 조회를 위해 이미지 ID를 (str)경로로 매핑
print("Creating image_id to image_path lookup map...")
image_id_to_path = {
    img['id']: str(FP_TRAIN_IMAGE_DIR / img['file_name'])
    for img in data_fp_train.get('images', [])
}
print(f"Map created for {len(image_id_to_path)} images.")

# --- 3. [핵심] Fashionpedia(Shop) 어노테이션 필터링 및 변환 ---
items_shop = []
annotations_fp = data_fp_train.get('annotations', [])

print(f"Processing {len(annotations_fp)} Fashionpedia (Shop) annotations...")

# tqdm을 사용하여 진행률 표시
for ann in tqdm(annotations_fp):
    fp_category_id = ann['category_id']
    
    # 1. '번역 규칙' (매핑 딕셔너리)에서 13종 타겟 인덱스 조회
    target_idx = fp_id_to_target_idx.get(fp_category_id)
    
    # 2. "무시"할 아이템(None)은 건너뛰기
    if target_idx is None:
        continue
        
    # 3. "사용"할 아이템만 리스트에 추가
    image_id = ann['image_id']
    image_path = image_id_to_path.get(image_id)
    
    # 이미지 경로가 존재하지 않으면 건너뛰기
    if image_path is None:
        continue
        
    # 재훈련에 필요한 정보만 추출하여 저장
    item_info = {
        'image_path': image_path,
        'box': ann['bbox'], # [x, y, w, h] 형식
        'category_id': int(target_idx), # [중요] 0~12 사이의 13종 레이블
        'source': 'shop' # [중요] H4 재훈련을 위한 출처
    }
    items_shop.append(item_info)

print(f"\nFashionpedia(Shop) 아이템 처리 완료!")
print(f"  - 원본 어노테이션 개수: {len(annotations_fp)}")
print(f"  - 13종 레이블로 매핑된 아이템 개수: {len(items_shop)}")

# --- 4. 다음 단계를 위해 변수 저장 ---
%store items_shop


/home/epistachio/miniconda3/envs/deeplearning/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Creating image_id to image_path lookup map...
Map created for 45623 images.
Processing 333401 Fashionpedia (Shop) annotations...


100%|██████████████████████████████████████████████████████████████████████| 333401/333401 [00:00<00:00, 3887707.74it/s]


✅ Fashionpedia(Shop) 아이템 처리 완료!
  - 원본 어노테이션 개수: 333401
  - 13종 레이블로 매핑된 아이템 개수: 75941
Stored 'items_shop' (list)


In [4]:
#cell 4

import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm # 진행률 표시
from PIL import Image

# --- 1. [Task 3]에서 저장한 변수 로드 ---
try:
    # 다음 단계(Task 5)에서 사용하기 위해 미리 로드
    %store -r items_shop
except Exception as e:
    print(f"[Fatal Error] '%store -r items_shop' failed. Task 3을 다시 실행해야 합니다: {e}")
    raise e

# --- 2. 경로 설정 (DeepFashion2 - YOLO 형식) ---
PROJECT_ROOT = Path.cwd()
DF2_DATA_DIR = PROJECT_ROOT / "data" / "deepfashion2_final"
DF2_IMAGES_DIR = DF2_DATA_DIR / "images" / "train"
DF2_LABELS_DIR = DF2_DATA_DIR / "labels" / "train" # [FIX] 'train' 폴더를 직접 타겟팅

print(f"Loading DeepFashion2 (Consumer) annotations from: {DF2_LABELS_DIR} (YOLO .txt format)")

# --- 3. [핵심] YOLO .txt 파일 파싱 및 변환 ---
items_consumer = []

# .txt 파일 목록 가져오기
label_files = sorted(DF2_LABELS_DIR.glob("*.txt"))
print(f"Found {len(label_files)} label files to process...")

if not label_files:
    print(f"[Fatal Error] '{DF2_LABELS_DIR}'에서 .txt 파일을 찾을 수 없습니다.")
    print("경로가 올바른지 다시 확인해 주세요.")
else:
    for label_path in tqdm(label_files):
        # 1. 이미지 경로 생성 (예: 000001.txt -> 000001.jpg)
        image_name = label_path.stem + ".jpg"
        image_path = DF2_IMAGES_DIR / image_name
        
        # 2. 원본 이미지 파일이 없으면 건너뛰기
        if not image_path.exists():
            continue
            
        # 3. BBox 변환을 위해 원본 이미지 크기 로드
        try:
            with Image.open(image_path) as img:
                img_w, img_h = img.size
        except Exception:
            continue # 손상된 이미지 건너뛰기
            
        # 4. .txt 파일 읽기
        with open(label_path, 'r') as f:
            lines = f.readlines()
            
        for line in lines:
            try:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue # 형식이 잘못된 라인 건너뛰기

                # 5. [중요] YOLO -> [x,y,w,h] 변환
                # (YOLO는 category_id가 이미 0-12 입니다)
                target_idx = int(parts[0]) 
                x_center_norm = float(parts[1])
                y_center_norm = float(parts[2])
                width_norm = float(parts[3])
                height_norm = float(parts[4])
                
                # 픽셀 좌표로 변환
                w_abs = width_norm * img_w
                h_abs = height_norm * img_h
                x_abs = (x_center_norm * img_w) - (w_abs / 2)
                y_abs = (y_center_norm * img_h) - (h_abs / 2)
                
                box_xywh = [x_abs, y_abs, w_abs, h_abs]
                
                item_info = {
                    'image_path': str(image_path),
                    'box': box_xywh,
                    'category_id': target_idx, # 0~12 사이의 13종 레이블
                    'source': 'consumer' # [중요] H4 재훈련을 위한 출처
                }
                items_consumer.append(item_info)
                
            except Exception:
                continue # 손상된 라인 건너뛰기

    print(f"\nDeepFashion2(Consumer) 아이템 처리 완료!")
    print(f"  - 원본 .txt 파일 개수: {len(label_files)}")
    print(f"  - 13종 레이블로 매핑된 아이템 개수: {len(items_consumer)}")

    # --- 4. 다음 단계를 위해 변수 저장 ---
    %store items_consumer

Loading DeepFashion2 (Consumer) annotations from: /home/epistachio/projects/fashion_ai/data/deepfashion2_final/labels/train (YOLO .txt format)
Found 191961 label files to process...


100%|█████████████████████████████████████████████████████████████████████████| 191961/191961 [01:39<00:00, 1922.84it/s]



✅ DeepFashion2(Consumer) 아이템 처리 완료!
  - 원본 .txt 파일 개수: 191961
  - 13종 레이블로 매핑된 아이템 개수: 312186
Stored 'items_consumer' (list)


In [5]:
#cell 5

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pathlib import Path

# --- 1. [Task 3, 4]에서 저장한 변수 로드 ---
try:
    %store -r items_shop
    %store -r items_consumer
    %store -r TARGET_LABELS # Task 2에서 정의한 13종 레이블
except Exception as e:
    print(f"[Fatal Error] '%store -r' failed. Task 3, 4를 다시 실행해야 합니다: {e}")
    raise e

print(f"Loaded {len(items_shop)} 'shop' items.")
print(f"Loaded {len(items_consumer)} 'consumer' items.")

# --- 2. [핵심] 두 리스트를 하나로 결합 ---
items_mixed = items_shop + items_consumer
total_items = len(items_mixed)
print(f"Total mixed items: {total_items}") # 약 388,127개

# --- 3. Pandas DataFrame으로 변환 ---
print("Converting to Pandas DataFrame...")
df_mixed = pd.DataFrame(items_mixed)

# --- 4. [중요] 훈련/검증용 데이터셋 분할 ---
# (Stratified Split: 'source'와 'category_id' 비율을 유지하며 분할)
print("Splitting into Train (90%) and Validation (10%) sets...")

# 'source'와 'category_id'를 조합하여 고유한 키 생성 (예: 'shop_0', 'consumer_7')
df_mixed['strata_key'] = df_mixed['source'] + '_' + df_mixed['category_id'].astype(str)

# 90% 훈련, 10% 검증
train_df, val_df = train_test_split(
    df_mixed,
    test_size=0.1,
    random_state=42,
    stratify=df_mixed['strata_key'] # 이 키를 기준으로 비율을 맞춰 분할
)

print(f"Split complete.")
print(f"  - Train items: {len(train_df)}")
print(f"  - Val items:   {len(val_df)}")

# --- 5. 최종 "마스터 CSV" 파일로 저장 ---
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True) # data 폴더가 없다면 생성

# 이 파일들이 재훈련의 기반이 됩니다.
TRAIN_MIXED_CSV_PATH = DATA_DIR / "mixed_dataset_train.csv"
VAL_MIXED_CSV_PATH = DATA_DIR / "mixed_dataset_val.csv"

try:
    train_df.to_csv(TRAIN_MIXED_CSV_PATH, index=False)
    val_df.to_csv(VAL_MIXED_CSV_PATH, index=False)
    
    print(f"\nTrain CSV saved to: {TRAIN_MIXED_CSV_PATH}")
    print(f"Val CSV saved to:   {VAL_MIXED_CSV_PATH}")

    # --- 6. 다음 단계를 위해 변수 저장 ---
    %store train_df
    %store val_df
    %store TARGET_LABELS

    print("\n[Phase 1: 데이터 준비]가 성공적으로 완료되었습니다!")
    
except Exception as e:
    print(f"\n[Error] CSV 파일 저장 중 오류 발생: {e}")
    


no stored variable or alias #
no stored variable or alias Task
no stored variable or alias 2에서
no stored variable or alias 정의한
no stored variable or alias 13종
no stored variable or alias 레이블
Loaded 75941 'shop' items.
Loaded 312186 'consumer' items.
✅ Total mixed items: 388127
Converting to Pandas DataFrame...
Splitting into Train (90%) and Validation (10%) sets...
Split complete.
  - Train items: 349314
  - Val items:   38813

✅ Train CSV saved to: /home/epistachio/projects/fashion_ai/data/mixed_dataset_train.csv
✅ Val CSV saved to:   /home/epistachio/projects/fashion_ai/data/mixed_dataset_val.csv
Stored 'train_df' (DataFrame)
Stored 'val_df' (DataFrame)
Stored 'TARGET_LABELS' (list)

[Phase 1: 데이터 준비]가 성공적으로 완료되었습니다!


In [6]:
#cell 6

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
from pathlib import Path

# --- 1. [Task 5]에서 저장한 변수 로드 ---
try:
    %store -r train_df
    %store -r val_df
    %store -r TARGET_LABELS
    print("Loaded 'train_df', 'val_df', and 'TARGET_LABELS' from storage.")
except Exception as e:
    print(f"[Fatal Error] '%store -r' failed. Task 5를 다시 실행해야 합니다: {e}")
    # 만약 %store가 실패하면, CSV에서 직접 로드
    PROJECT_ROOT = Path.cwd()
    DATA_DIR = PROJECT_ROOT / "data"
    TRAIN_MIXED_CSV_PATH = DATA_DIR / "mixed_dataset_train.csv"
    VAL_MIXED_CSV_PATH = DATA_DIR / "mixed_dataset_val.csv"
    
    print("Loading DataFrames from CSV files as fallback...")
    train_df = pd.read_csv(TRAIN_MIXED_CSV_PATH)
    val_df = pd.read_csv(VAL_MIXED_CSV_PATH)
    TARGET_LABELS = [
        'short sleeve top', 'long sleeve top', 'short sleeve outwear',
        'long sleeve outwear', 'vest', 'sling', 'shorts',
        'trousers', 'skirt', 'short sleeve dress',
        'long sleeve dress', 'vest dress', 'sling dress'
    ]


# --- 2. 아이템 크롭 함수 ---
def crop_item(image_path, box_xywh):
    """
    전체 이미지 경로와 [x, y, w, h] 박스를 받아 아이템 이미지를 잘라 리턴
    """
    try:
        full_image = Image.open(image_path).convert("RGB")
        # 'box' 컬럼이 문자열(string)로 저장되었을 경우, list로 변환
        if isinstance(box_xywh, str):
            box_xywh = eval(box_xywh)
            
        x, y, w, h = box_xywh
        # [x1, y1, x2, y2] 형식으로 변환
        return full_image.crop((x, y, x + w, y + h))
    except Exception as e:
        # 오류 발생 시 빈 224x224 검은색 이미지 반환
        return Image.new("RGB", (224, 224), (0, 0, 0))

# --- 3. "혼합 데이터셋"용 H3 Dataset 클래스 ---
class H3MixedDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform
        print("Converting 'box' column from string to list (if needed)...")
        # CSV에서 다시 로드한 경우 'box'가 문자열일 수 있음
        if not self.df.empty and isinstance(self.df.iloc[0]['box'], str):
             self.df['box'] = self.df['box'].apply(eval)
        print("'box' column is ready.")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        item_info = self.df.iloc[idx]
        item_image = crop_item(item_info['image_path'], item_info['box'])
        label = torch.tensor(item_info['category_id'], dtype=torch.long)
        
        if self.transform:
            item_image = self.transform(item_image)
            
        return item_image, label

# --- 4. CLIP 모델에 맞는 이미지 전처리 ---
preprocess_h3 = T.Compose([
    T.Resize((224, 224), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
])

# --- 5. 데이터로더 생성 ---
print("\nCreating H3-Mixed DataLoaders...")
try:
    train_dataset_h3_mixed = H3MixedDataset(train_df, transform=preprocess_h3)
    
    # [FIX] 속도와 안정성을 위해 num_workers=2로 설정
    train_loader_h3_mixed = DataLoader(
        train_dataset_h3_mixed, 
        batch_size=256, 
        shuffle=True, 
        num_workers=2,  # <-- 수정됨 (0 -> 2)
        pin_memory=False # False로 유지 (안정성)
    )

    val_dataset_h3_mixed = H3MixedDataset(val_df, transform=preprocess_h3)
    val_loader_h3_mixed = DataLoader(
        val_dataset_h3_mixed, 
        batch_size=512, 
        shuffle=False, 
        num_workers=2,  # <-- 수정됨 (0 -> 2)
        pin_memory=False # False로 유지
    )

    print(f"\nH3 재훈련용 데이터로더 생성 완료. (Balanced Mode: num_workers=2)")
    print(f"  - H3 Train batches: {len(train_loader_h3_mixed)}")
    print(f"  - H3 Val batches:   {len(val_loader_h3_mixed)}")

    # --- 6. 다음 단계를 위해 변수 저장 ---
    %store train_loader_h3_mixed
    %store val_loader_h3_mixed

except Exception as e:
    print(f"\n[Fatal Error] 데이터로더 생성 중 오류 발생: {e}")

Loaded 'train_df', 'val_df', and 'TARGET_LABELS' from storage.

Creating H3-Mixed DataLoaders...
Converting 'box' column from string to list (if needed)...
'box' column is ready.
Converting 'box' column from string to list (if needed)...
'box' column is ready.

✅ H3 재훈련용 데이터로더 생성 완료. (Balanced Mode: num_workers=2)
  - H3 Train batches: 1365
  - H3 Val batches:   76
Stored 'train_loader_h3_mixed' (DataLoader)
Stored 'val_loader_h3_mixed' (DataLoader)


In [7]:
#cell 7

import torch
import torch.nn as nn
import torch.optim as optim
import open_clip
import time
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score

# --- 1. [Task 6]에서 저장한 변수 로드 ---
try:
    %store -r train_loader_h3_mixed
    %store -r val_loader_h3_mixed
    %store -r TARGET_LABELS
    print("Loaded DataLoaders and TARGET_LABELS from storage.")
except Exception as e:
    print(f"[Fatal Error] '%store -r' failed. Task 6 (Balanced)을 다시 실행해야 합니다: {e}")
    raise e

# --- 2. 기본 설정 및 OpenCLIP 모델 로드 ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_H3_CLASSES = len(TARGET_LABELS) # 13

print(f"Initializing models on {DEVICE}...")
print(f"Target H3 classes: {N_H3_CLASSES}")

try:
    # H3 모델의 백본(Backbone)으로 사용할 CLIP 모델 로드
    clip_model, _, _ = open_clip.create_model_and_transforms(
        "ViT-B-32", 
        pretrained="laion2b_s34b_b79k", 
        device=DEVICE
    )
    clip_model.eval()
    EMB_DIM = clip_model.visual.output_dim
    print(f"OpenCLIP base model loaded. EMB_DIM={EMB_DIM}")
except Exception as e:
    print(f"[Fatal Error] Failed to load OpenCLIP model: {e}")
    raise e

# --- 3. H3 분류 모델 아키텍처 정의 ---
class H3Classifier(nn.Module):
    def __init__(self, emb_dim=EMB_DIM, num_classes=N_H3_CLASSES):
        super().__init__()
        self.encoder = clip_model.visual # CLIP 인코더
        
        # CLIP 인코더는 동결(freeze)
        for param in self.encoder.parameters():
            param.requires_grad = False
            
        # [중요] 훈련 가능한 분류 헤드
        self.classifier_head = nn.Sequential(
            nn.Linear(emb_dim, 512), 
            nn.ReLU(), 
            nn.Dropout(0.5), 
            nn.Linear(512, num_classes)
        ).to(DEVICE)

    def forward(self, x):
        with torch.no_grad():
            embedding = self.encoder(x)
        logits = self.classifier_head(embedding)
        return logits

# --- 4. 평가 함수 정의 ---
@torch.no_grad()
def evaluate_h3(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    
    for images, labels in tqdm(loader, desc="Evaluating", leave=False):
        images = images.to(DEVICE)
        
        logits = model(images)
        preds = torch.argmax(logits, dim=1)
        
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())
        
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    
    accuracy = accuracy_score(all_labels, all_preds)
    return {'accuracy': accuracy}

# --- 5. [핵심] H3 재훈련 루프 실행 ---
print("\n--- Starting H3 Model Retraining (Mixed Dataset) ---")

model_h3_mixed = H3Classifier().to(DEVICE)
optimizer = optim.Adam(model_h3_mixed.classifier_head.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()

# [FIX] 정확도를 높이기 위해 1 -> 3 에포크로 늘림
epochs = 3 
best_accuracy = 0.0
H3_MIXED_CKPT_PATH = ""
CHECKPOINT_DIR = Path.cwd() / "checkpoints_mixed"
CHECKPOINT_DIR.mkdir(exist_ok=True)

for epoch in range(1, epochs + 1):
    # --- 훈련 ---
    model_h3_mixed.train()
    start_time = time.time()
    running_loss = 0.0
    
    train_progress = tqdm(train_loader_h3_mixed, desc=f"Epoch {epoch}/{epochs} Training")
    for i, (images, labels) in enumerate(train_progress):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        
        logits = model_h3_mixed(images)
        loss = loss_fn(logits, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        train_progress.set_postfix(loss=loss.item())
            
    avg_train_loss = running_loss / len(train_loader_h3_mixed)

    # --- 평가 ---
    print(f"\nEvaluating on validation set...")
    val_metrics = evaluate_h3(model_h3_mixed, val_loader_h3_mixed)
    
    end_time = time.time()
    epoch_duration = end_time - start_time
    
    print(f"\nEpoch {epoch} complete.")
    print(f"  - Duration: {epoch_duration:.2f}s")
    print(f"  - Average Train Loss: {avg_train_loss:.4f}")
    print(f"  - Validation Accuracy: {val_metrics['accuracy']:.4f}")

    # --- 최고 성능 모델 저장 ---
    if val_metrics['accuracy'] > best_accuracy:
        best_accuracy = val_metrics['accuracy']
        H3_MIXED_CKPT_PATH = CHECKPOINT_DIR / f"h3_mixed_best_epoch{epoch}_acc{best_accuracy:.3f}.pt"
        torch.save(model_h3_mixed.classifier_head.state_dict(), H3_MIXED_CKPT_PATH)
        print(f"  >>> New best H3 model saved to: {H3_MIXED_CKPT_PATH}\n")

print("H3 model retraining finished!")
if H3_MIXED_CKPT_PATH:
    print(f"Best H3 (Mixed) model checkpoint is at: {H3_MIXED_CKPT_PATH}")
    %store H3_MIXED_CKPT_PATH
else:
    print("No H3 model was saved.")

Loaded DataLoaders and TARGET_LABELS from storage.
Initializing models on cuda...
Target H3 classes: 13
OpenCLIP base model loaded. EMB_DIM=512

--- Starting H3 Model Retraining (Mixed Dataset) ---


Epoch 1/3 Training: 100%|███████████████████████████████████████████████| 1365/1365 [07:18<00:00,  3.11it/s, loss=0.894]



Evaluating on validation set...



Epoch 1 complete.
  - Duration: 488.38s
  - Average Train Loss: 1.1596
  - Validation Accuracy: 0.6634
  >>> New best H3 model saved to: /home/epistachio/projects/fashion_ai/checkpoints_mixed/h3_mixed_best_epoch1_acc0.663.pt



Epoch 2/3 Training: 100%|███████████████████████████████████████████████| 1365/1365 [07:10<00:00,  3.17it/s, loss=0.864]



Evaluating on validation set...



Epoch 2 complete.
  - Duration: 482.70s
  - Average Train Loss: 0.9307
  - Validation Accuracy: 0.6727
  >>> New best H3 model saved to: /home/epistachio/projects/fashion_ai/checkpoints_mixed/h3_mixed_best_epoch2_acc0.673.pt



Epoch 3/3 Training: 100%|███████████████████████████████████████████████| 1365/1365 [07:06<00:00,  3.20it/s, loss=0.764]



Evaluating on validation set...



Epoch 3 complete.
  - Duration: 473.91s
  - Average Train Loss: 0.8971
  - Validation Accuracy: 0.6780
  >>> New best H3 model saved to: /home/epistachio/projects/fashion_ai/checkpoints_mixed/h3_mixed_best_epoch3_acc0.678.pt

H3 model retraining finished!
Best H3 (Mixed) model checkpoint is at: /home/epistachio/projects/fashion_ai/checkpoints_mixed/h3_mixed_best_epoch3_acc0.678.pt
Stored 'H3_MIXED_CKPT_PATH' (PosixPath)


In [5]:
#cell 8

import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image
import os
import shutil

# --- 1. [Task 5]에서 저장한 변수 로드 ---
try:
    %store -r train_df
    %store -r val_df
    %store -r TARGET_LABELS # 13종 레이블 리스트
    print("Loaded 'train_df', 'val_df', and 'TARGET_LABELS' from storage.")
except Exception as e:
    print(f"[Fatal Error] '%store -r' failed. Task 5를 다시 실행해야 합니다: {e}")
    raise e

# --- 2. YOLO 훈련용 디렉토리 생성 ---
PROJECT_ROOT = Path.cwd()
YOLO_DATA_DIR = PROJECT_ROOT / "data" / "yolo_mixed_dataset"
YOLO_TRAIN_IMAGES = YOLO_DATA_DIR / "images" / "train"
YOLO_TRAIN_LABELS = YOLO_DATA_DIR / "labels" / "train"
YOLO_VAL_IMAGES = YOLO_DATA_DIR / "images" / "val"
YOLO_VAL_LABELS = YOLO_DATA_DIR / "labels" / "val"

# os.makedirs(..., exist_ok=True)와 동일
YOLO_TRAIN_IMAGES.mkdir(parents=True, exist_ok=True)
YOLO_TRAIN_LABELS.mkdir(parents=True, exist_ok=True)
YOLO_VAL_IMAGES.mkdir(parents=True, exist_ok=True)
YOLO_VAL_LABELS.mkdir(parents=True, exist_ok=True)

print(f"YOLO 훈련용 디렉토리 생성 완료: {YOLO_DATA_DIR}")

# --- 3. [핵심] DataFrame -> YOLO .txt 변환 함수 ---
def convert_to_yolo_format(df, image_dir, label_dir):
    """
    DataFrame을 입력받아, 이미지/라벨 폴더에 YOLO 형식으로 저장합니다.
    """
    print(f"Converting {len(df)} items to YOLO format in {label_dir}...")
    
    image_groups = {}
    
    print("Grouping items by image_path...")
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Grouping"):
        img_path_str = row['image_path']
        if img_path_str not in image_groups:
            try:
                # [FIX] CSV에서 로드할 때 'box'가 문자열일 수 있으므로 eval
                if isinstance(row['box'], str):
                    row['box'] = eval(row['box'])
                    
                with Image.open(img_path_str) as img:
                    img_w, img_h = img.size
                image_groups[img_path_str] = {'size': (img_w, img_h), 'items': []}
            except Exception:
                continue 
                
        image_groups[img_path_str]['items'].append(row)

    print(f"Processing {len(image_groups)} unique images...")
    processed_count = 0
    
    for img_path_str, data in tqdm(image_groups.items(), desc="Writing YOLO files"):
        img_path = Path(img_path_str)
        img_w, img_h = data['size']
        
        # 새 이미지 파일명 (원본 파일명 사용)
        new_image_path = image_dir / img_path.name
        
        # 라벨 파일명
        label_name = img_path.stem + ".txt"
        label_save_path = label_dir / label_name
        
        # [FIX] 원본 이미지를 새 훈련 폴더로 "복사" (심볼릭 링크 대신)
        # (이미 존재하지 않는 경우에만 복사)
        if not new_image_path.exists():
            try:
                shutil.copyfile(img_path_str, new_image_path)
            except Exception as e:
                # print(f"Warning: Failed to copy {img_path_str}: {e}")
                continue # 이미지 복사 실패 시 .txt도 생성 안 함

        yolo_lines = []
        for item in data['items']:
            category_id = item['category_id'] # 0~12
            
            # [FIX] CSV에서 로드할 때 'box'가 문자열일 수 있으므로 eval
            if isinstance(item['box'], str):
                box_xywh = eval(item['box'])
            else:
                box_xywh = item['box']
            
            x, y, w, h = box_xywh # [x, y, w, h] 픽셀
            
            x_center_norm = (x + w / 2) / img_w
            y_center_norm = (y + h / 2) / img_h
            width_norm = w / img_w
            height_norm = h / img_h
            
            yolo_lines.append(f"{category_id} {x_center_norm} {y_center_norm} {width_norm} {height_norm}\n")
        
        with open(label_save_path, 'w') as f:
            f.writelines(yolo_lines)
            
        processed_count += 1

    return processed_count

# --- 4. 변환 실행 ---
# (주의: 이 작업은 데이터 양에 따라 30분~1시간 이상 소요될 수 있습니다)
print("\n--- [START] Converting Train Set ---")
train_images_processed = convert_to_yolo_format(train_df, YOLO_TRAIN_IMAGES, YOLO_TRAIN_LABELS)
print(f"Train Set 변환 완료. {train_images_processed}개의 .txt 및 이미지 파일 처리.")

print("\n--- [START] Converting Validation Set ---")
val_images_processed = convert_to_yolo_format(val_df, YOLO_VAL_IMAGES, YOLO_VAL_LABELS)
print(f"Validation Set 변환 완료. {val_images_processed}개의 .txt 및 이미지 파일 처리.")

# --- 5. YOLOv8 'dataset.yaml' 파일 생성 ---
YAML_PATH = YOLO_DATA_DIR / "mixed_dataset.yaml"
print(f"\nCreating dataset YAML file at: {YAML_PATH}")

yaml_content = f"""
# YOLOv8 훈련용 '혼합 데이터셋' (Fashionpedia + DeepFashion2)
# 경로: {YOLO_DATA_DIR.resolve()}

path: {str(YOLO_DATA_DIR.resolve())}
train: images/train
val: images/val
# test: (필요시 추가)

# 클래스 이름 (13종)
names:
"""
for idx, name in enumerate(TARGET_LABELS):
    yaml_content += f"  {idx}: {name}\n"

with open(YAML_PATH, 'w') as f:
    f.write(yaml_content)

print(f"'mixed_dataset.yaml' 파일 생성 완료.")
print("\n[Phase 2 - Step 2/3] YOLO 데이터 준비가 완료되었습니다!")

# --- 6. 다음 단계를 위해 변수 저장 ---
%store YAML_PATH
%store YOLO_DATA_DIR

no stored variable or alias #
no stored variable or alias 13종
no stored variable or alias 레이블
no stored variable or alias 리스트
Loaded 'train_df', 'val_df', and 'TARGET_LABELS' from storage.
YOLO 훈련용 디렉토리 생성 완료: /home/epistachio/projects/fashion_ai/data/yolo_mixed_dataset

--- [START] Converting Train Set ---
Converting 349314 items to YOLO format in /home/epistachio/projects/fashion_ai/data/yolo_mixed_dataset/labels/train...
Grouping items by image_path...


Grouping: 100%|██████████████████████████████████████████████████████████████| 349314/349314 [00:14<00:00, 24297.41it/s]


Processing 183189 unique images...


Writing YOLO files: 100%|████████████████████████████████████████████████████| 183189/183189 [00:09<00:00, 19836.41it/s]


✅ Train Set 변환 완료. 183189개의 .txt 및 이미지 파일 처리.

--- [START] Converting Validation Set ---
Converting 38813 items to YOLO format in /home/epistachio/projects/fashion_ai/data/yolo_mixed_dataset/labels/val...
Grouping items by image_path...


Grouping: 100%|████████████████████████████████████████████████████████████████| 38813/38813 [00:01<00:00, 21891.35it/s]


Processing 29926 unique images...


Writing YOLO files: 100%|██████████████████████████████████████████████████████| 29926/29926 [00:01<00:00, 22526.87it/s]

✅ Validation Set 변환 완료. 29926개의 .txt 및 이미지 파일 처리.

Creating dataset YAML file at: /home/epistachio/projects/fashion_ai/data/yolo_mixed_dataset/mixed_dataset.yaml
✅ 'mixed_dataset.yaml' 파일 생성 완료.

[Phase 2 - Step 2/3] YOLO 데이터 준비가 완료되었습니다!
Stored 'YAML_PATH' (PosixPath)
Stored 'YOLO_DATA_DIR' (PosixPath)


In [6]:
#cell 9

import torch
import os
from pathlib import Path
from ultralytics import YOLO

# --- 1. [Task 8]에서 저장한 변수 로드 ---
try:
    %store -r YAML_PATH
    print(f"Loaded 'YAML_PATH': {YAML_PATH}")
except Exception as e:
    print(f"[Fatal Error] '%store -r' failed. Task 8을 다시 실행해야 합니다: {e}")
    # Task 8에서 로드를 못했을 경우를 대비한 복구 코드
    if 'YAML_PATH' not in locals():
        PROJECT_ROOT = Path.cwd()
        YOLO_DATA_DIR = PROJECT_ROOT / "data" / "yolo_mixed_dataset"
        YAML_PATH = YOLO_DATA_DIR / "mixed_dataset.yaml"
        if not YAML_PATH.exists():
            print(f"[Fatal Error] YAML_PATH 변수를 복구할 수 없고, {YAML_PATH} 파일도 없습니다.")
            print("Task 8을 다시 실행해 주세요.")
            raise FileNotFoundError(str(YAML_PATH))
        else:
            print(f"Recovered 'YAML_PATH': {YAML_PATH}")


# --- 2. 기본 설정 ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PROJECT_ROOT = Path.cwd()
YOLO_MIXED_CKPT_DIR = PROJECT_ROOT / "checkpoints_mixed" # 새 모델 저장 위치
YOLO_MIXED_CKPT_DIR.mkdir(exist_ok=True)

# --- 3. [핵심] YOLOv8 재훈련 실행 ---
print(f"\n--- Starting YOLOv8 Model Retraining (Mixed Dataset) ---")
print(f"Using dataset config: {YAML_PATH}")
print(f"Models will be saved to: {YOLO_MIXED_CKPT_DIR}")

try:
    # 1. 훈련을 시작할 기반 모델 로드
    base_model_path = "yolo11n.pt"
    if not Path(base_model_path).exists():
        print(f"Warning: {base_model_path} not found. Using 'yolov8n.pt' as base.")
        base_model_path = "yolov8n.pt" 
    
    print(f"Using base model: {base_model_path}")
    model = YOLO(base_model_path)
    
    # 2. 훈련 실행
    results = model.train(
        data=str(YAML_PATH),
        epochs=3,
        batch=16,       # VRAM은 16GB로 충분하므로 16 유지
        device=0 if DEVICE == "cuda" else "cpu",
        project=str(YOLO_MIXED_CKPT_DIR), 
        name="yolo_mixed_run",
        
        # [FIX] 메모리 충돌 방지를 위해 H3와 동일하게 2로 고정
        workers=2,
        
        # [FIX] 이전 훈련 로그에 이어서 저장 (기존 yolo_mixed_run 폴더 덮어쓰기)
        exist_ok=True 
    )
    
    print("\nYOLO model retraining finished!")

    # --- 4. 훈련된 가중치 경로 저장 ---
    YOLO_MIXED_BEST_PT = results.save_dir / "weights" / "best.pt"
    
    if YOLO_MIXED_BEST_PT.exists():
        print(f"Best YOLO (Mixed) model checkpoint is at: {YOLO_MIXED_BEST_PT}")
        %store YOLO_MIXED_BEST_PT
    else:
        print("[Error] Could not find the saved 'best.pt' file.")

except Exception as e:
    print(f"\n[Fatal Error] YOLO 훈련 중 오류 발생: {e}")
    if "CUDA out of memory" in str(e):
        print("\n[Tip] 'CUDA out of memory' 오류가 발생했습니다.")
        print("    -> 'batch=16' 값을 8 또는 4로 줄여서 다시 시도해 보세요.")
    elif "No such file or directory" in str(e) and "yolo" in str(e):
         print("\n[Tip] 'yolo11n.pt' 또는 'yolov8n.pt' 파일을 찾을 수 없습니다.")
         print("    -> E2E 파이프라인 노트북 폴더에 해당 파일이 있는지 확인하세요.")

Loaded 'YAML_PATH': /home/epistachio/projects/fashion_ai/data/yolo_mixed_dataset/mixed_dataset.yaml

--- Starting YOLOv8 Model Retraining (Mixed Dataset) ---
Using dataset config: /home/epistachio/projects/fashion_ai/data/yolo_mixed_dataset/mixed_dataset.yaml
Models will be saved to: /home/epistachio/projects/fashion_ai/checkpoints_mixed
Using base model: yolo11n.pt
New https://pypi.org/project/ultralytics/8.3.226 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.225 🚀 Python-3.11.14 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 16303MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/epistachio/projects/fashion_ai/data/yolo_mixed_dataset/mixed_dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0

In [ ]:
#cell 10

